# 00 — DG38 Preflight, Kiểm Tra GPU & Dự Báo Ngân Sách Colab Pro (200.45 CU)

**Mục đích của Notebook này:**
1. Mount Google Drive, tự động giải nén gói mã nguồn (`dg38_source_code_and_manifests.zip`), gói checkpoints tham chiếu (`dg38_reference_checkpoints.zip`), gói SQLite cache (`dg38_crop_ablation_val_bundle.zip`) vào `DRIVE_ROOT/code` (nếu chưa giải nén) và giải nén nhanh dữ liệu Train/Val sang ổ SSD cục bộ của Colab (`/content/dg38_data`) để tránh nghẽn I/O.
2. Tự động nhận diện GPU hiện tại (**NVIDIA L4, A100, G4, T4...**), kiểm tra CUDA, PyTorch, Total/Available VRAM, BF16/FP16 và đề xuất cấu hình tối ưu.
3. Xác minh toàn vẹn SHA-256 của các manifest và checkpoints gốc:
   - **Classification**: `9,807` Train / `1,226` Val (`dg38_cls_train_val.zip`) và `1,226` Test tách riêng (`dg38_cls_test_locked.zip`).
   - **Detection**: `8,289` Train (`18,345` boxes) / `1,743` Clean Validation (`3,932` boxes) (`dg38_det_train_val_clean.zip`). *(Gói Test `871` ảnh / `1,884` boxes `dg38_det_test_locked.zip` được tách riêng, không nạp trong bước huấn luyện/chọn mô hình)*.
   - **Checkpoints gốc**: `checkpoints/dinov2_garbage_final.pth` và `detection/checkpoints_wastedetect/best_model.pt`.
4. Nhập **Usage Rate (CU/giờ)** thực tế hiển thị trên giao diện Google Colab (`Runtime -> View resources`) và ghi nhận số dư CU hiện tại vào `cu_budget_ledger.json`.
5. Chạy **Smoke Test** cho Classification, Detection (YOLO + Deformable DETR `MSDeformAttn`) trước khi chạy bất kỳ thí nghiệm nào.

In [ ]:
# [CELL 1] Mount Google Drive & Giải nén gói mã nguồn / checkpoints trên Drive (nếu cần)
import os
import sys
import shutil
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

DRIVE_ROOT = Path(os.environ.get("DG38_DRIVE_ROOT", "/content/drive/MyDrive/DG38_Colab_Package"))
ZIPS_DIR = DRIVE_ROOT / "zips"
CODE_DIR = DRIVE_ROOT / "code"
ARTIFACTS_DIR = DRIVE_ROOT / "artifacts"
LOCAL_WORK_DIR = Path("/content/dg38_workspace")
LOCAL_DATA_DIR = Path("/content/dg38_data")

LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_ROOT.exists():
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    CODE_DIR.mkdir(parents=True, exist_ok=True)

    # Tự động giải nén mã nguồn + manifests vào DRIVE_ROOT/code nếu chưa có model.py
    src_zip = ZIPS_DIR / "dg38_source_code_and_manifests.zip"
    if src_zip.exists() and not (CODE_DIR / "model.py").exists():
        print(f"Đang giải nén mã nguồn {src_zip} -> {CODE_DIR} ...")
        with zipfile.ZipFile(src_zip, "r") as zf:
            zf.extractall(CODE_DIR)

    # Tự động giải nén checkpoints gốc vào DRIVE_ROOT/code nếu chưa có
    ckpt_zip = ZIPS_DIR / "dg38_reference_checkpoints.zip"
    if ckpt_zip.exists() and not (CODE_DIR / "checkpoints" / "dinov2_garbage_final.pth").exists():
        print(f"Đang giải nén checkpoints tham chiếu {ckpt_zip} -> {CODE_DIR} ...")
        with zipfile.ZipFile(ckpt_zip, "r") as zf:
            zf.extractall(CODE_DIR)

    # Tự động giải nén gói SQLite cache cho Crop Ablation vào DRIVE_ROOT/code nếu chưa có
    crop_zip = ZIPS_DIR / "dg38_crop_ablation_val_bundle.zip"
    if crop_zip.exists() and not (CODE_DIR / "evaluation_runs" / "val_grid_search_01" / "cache" / "predictions.sqlite3").exists():
        print(f"Đang giải nén gói Crop Ablation cache {crop_zip} -> {CODE_DIR} ...")
        with zipfile.ZipFile(crop_zip, "r") as zf:
            zf.extractall(CODE_DIR)

REPO_ROOT = CODE_DIR if (CODE_DIR / "model.py").exists() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"IN_COLAB       : {IN_COLAB}")
print(f"DRIVE_ROOT     : {DRIVE_ROOT} (exists={DRIVE_ROOT.exists()})")
print(f"REPO_ROOT      : {REPO_ROOT} (model.py exists={(REPO_ROOT / 'model.py').exists()})")
print(f"LOCAL_DATA_DIR : {LOCAL_DATA_DIR}")

In [ ]:
# [CELL 2] Cài đặt các thư viện cần thiết trên Colab
!pip install -q timm transformers accelerate ultralytics pycocotools scikit-learn pandas pillow

In [ ]:
# [CELL 3] Kiểm tra phần cứng GPU động (L4 / A100 / G4) & đề xuất cấu hình
import json
from evaluation_runs.dg38_additional_experiments.gpu_runtime_manager import (
    inspect_runtime,
    recommend_execution_config,
)

rt_info = inspect_runtime()
print("=== THÔNG TIN PHẦN CỨNG GPU HIỆN TẠI ===")
print(json.dumps(rt_info, indent=2, ensure_ascii=False))

print("\n=== ĐỀ XUẤT CẤU HÌNH THEO NHIỆM VỤ TRÊN GPU HIỆN TẠI ===")
for task in [
    "smoke_test",
    "resnet50_classification",
    "yolo11s_detection",
    "crop_preprocessing_ablation",
    "dinov2_vitb14_finetune",
    "deformable_detr_training",
]:
    rec = recommend_execution_config(task, rt_info)
    print(
        f"[{task:27s}] GPU={rec['detected_gpu_family']:6s} | AMP={rec['amp_dtype']:8s} | "
        f"Batch={rec['physical_batch_size']:2d} x Accum={rec['grad_accum_steps']} (Eff={rec['effective_batch_size']:2d}) | "
        f"Khuyến nghị: {rec['recommended_primary_gpu']}"
    )

In [ ]:
# [CELL 4] Giải nén dữ liệu Train/Val sang ổ SSD `/content/dg38_data` & Kiểm tra SHA-256 Manifest + Checkpoints
from evaluation_runs.dg38_additional_experiments.prepare_colab_packages import sha256_file

EXP_DIR = REPO_ROOT / "evaluation_runs" / "dg38_additional_experiments"
MANIFEST_DIR = EXP_DIR / "manifests"

for zip_name, target_sub, sentinel_rel in [
    (
        "dg38_cls_train_val.zip",
        LOCAL_DATA_DIR / "classification",
        "standardized_256",
    ),
    (
        "dg38_det_train_val_clean.zip",
        LOCAL_DATA_DIR / "detection",
        "WasteDetect_Dataset/valid_clean_candidate/_annotations.coco.json",
    ),
]:
    zip_path = ZIPS_DIR / zip_name
    sentinel_path = target_sub / sentinel_rel
    if zip_path.exists() and not sentinel_path.exists():
        print(f"Đang giải nén {zip_path} -> {target_sub} ...")
        target_sub.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target_sub)
        print(f"[OK] Hoàn tất giải nén {zip_name}!")
    else:
        print(f"[INFO] {zip_name}: zip_exists={zip_path.exists()}, unpacked={sentinel_path.exists()}")

# Kiểm tra các đường dẫn quan trọng
cls_ckpt = REPO_ROOT / "checkpoints" / "dinov2_garbage_final.pth"
det_ckpt = REPO_ROOT / "detection" / "checkpoints_wastedetect" / "best_model.pt"
sqlite_cache = REPO_ROOT / "evaluation_runs" / "val_grid_search_01" / "cache" / "predictions.sqlite3"

for label, p in [
    ("Classifier Checkpoint (checkpoints/dinov2_garbage_final.pth)", cls_ckpt),
    ("Detector Checkpoint (detection/checkpoints_wastedetect/best_model.pt)", det_ckpt),
    ("Validation SQLite Cache (predictions.sqlite3)", sqlite_cache),
]:
    if p.exists():
        print(f"[OK] {label}: size={p.stat().st_size / (1024**2):.2f} MB | sha256={sha256_file(p)[:16]}...")
    else:
        print(f"[CẢNH BÁO] Chưa tìm thấy {label} tại {p}")

index_path = MANIFEST_DIR / "dataset_manifests_index.json"
if index_path.exists():
    idx_data = json.loads(index_path.read_text(encoding="utf-8"))
    print("\n=== THÔNG TIN MANIFEST ĐÃ KHOÁ (SHA-256) ===")
    print(json.dumps(idx_data, indent=2, ensure_ascii=False))
else:
    print(f"[CẢNH BÁO] Không tìm thấy {index_path}.")

In [ ]:
# [CELL 5] Nhập Usage Rate (CU/giờ) thực tế từ giao diện Colab & Đồng bộ Ngân sách
# HƯỚNG DẪN BẮT BUỘC: Mở menu Colab "Runtime -> View resources" để xem tốc độ tiêu thụ thực tế
# (Ví dụ tham khảo: L4 ~ 0.8 - 1.7 CU/h, A100 ~ 5.0 - 6.5 CU/h; vui lòng nhập đúng con số hiển thị trên Colab UI của bạn)
from evaluation_runs.dg38_additional_experiments.cu_budget_tracker import CUBudgetTracker

OBSERVED_CU_PER_HOUR = None     # <-- BẮT BUỘC: Thay None bằng số thực từ Colab UI (ví dụ: 1.48)
OBSERVED_REMAINING_CU = 200.45  # <-- Cập nhật số dư CU hiện tại trên giao diện Colab

if OBSERVED_CU_PER_HOUR is None or float(OBSERVED_CU_PER_HOUR) <= 0:
    raise ValueError(
        "Vui lòng mở 'Runtime -> View resources' trên Google Colab và nhập Usage Rate thực tế vào biến OBSERVED_CU_PER_HOUR (ví dụ: 1.48)!"
    )

LEDGER_PATH = EXP_DIR / "cu_budget_ledger.json"
tracker = CUBudgetTracker(LEDGER_PATH)
tracker.sync_user_reported_balance(
    remaining_cu_from_colab_ui=float(OBSERVED_REMAINING_CU),
    note=f"Đồng bộ từ giao diện Colab (GPU={rt_info['gpu_name']}, rate={OBSERVED_CU_PER_HOUR} CU/h)",
)
print(tracker.format_budget_table_markdown())

In [ ]:
# [CELL 6] Smoke Test: Kiểm tra Forward/Backward ResNet50 & Deformable DETR MSDeformAttn trên GPU hiện tại
import time
import torch
import torchvision.models as tv_models
from detection.dinov2_deformable_detr import MSDeformAttn

t0 = time.perf_counter()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Smoke test Torchvision ResNet50
m_rn50 = tv_models.resnet50(weights=None, num_classes=10).to(device)
x_dummy = torch.randn(4, 3, 224, 224, device=device)
out_rn50 = m_rn50(x_dummy)
loss_rn50 = out_rn50.sum()
loss_rn50.backward()
assert out_rn50.shape == (4, 10), f"Sai shape ResNet50: {out_rn50.shape}"
del m_rn50, out_rn50, loss_rn50

# 2. Smoke test Deformable DETR Pure-PyTorch MSDeformAttn
ms_attn = MSDeformAttn(d_model=256, n_levels=3, n_heads=8, n_points=4).to(device)
q = torch.randn(2, 100, 256, device=device)
ref_pts = torch.rand(2, 100, 3, 2, device=device)
spatial_shapes = torch.tensor([[36, 36], [18, 18], [9, 9]], dtype=torch.long, device=device)
total_tokens = int((spatial_shapes[:, 0] * spatial_shapes[:, 1]).sum().item())
src_feat = torch.randn(2, total_tokens, 256, device=device)
level_start = torch.tensor([0, 1296, 1296 + 324], dtype=torch.long, device=device)
out_attn = ms_attn(q, ref_pts, src_feat, spatial_shapes, level_start)
assert out_attn.shape == (2, 100, 256), f"Sai shape MSDeformAttn: {out_attn.shape}"
del ms_attn, q, ref_pts, src_feat, out_attn

if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed_s = time.perf_counter() - t0
print(f"[OK] Smoke test thành công trên {rt_info['gpu_name']} trong {elapsed_s:.2f}s!")